# Day 5 — Space Complexity, Recursion Stack, Python Traps

Time complexity asks "how long?". Space complexity asks **"how much extra memory?"** — and *extra* is the key word.

- **Auxiliary space** = only the scratch memory the algorithm creates (variables, temp lists, the recursion stack). Like the small tiffin *I* carry with my own tools.
- **Total space** = auxiliary + the input itself. The whole kitchen — my tiffin **plus** the customer's big dabba that was handed to me.

Interview convention: when someone says "do it in O(1) space", they mean **auxiliary** space. The input is always allowed to exist.

The demo cells below are small experiments. Before running each one, read the note above it and predict the output first.

## Demo 1 — Recursion uses memory even with no lists

`countdown` calls itself. Every call that has started but not finished must **wait** — Python parks it on the **call stack**, like plates stacked in a hostel mess: newest on top, and nothing below can leave until everything above is done.

**What to observe:** the `enter` lines print while the stack is *growing*, the `leave` lines print while it is *shrinking* — in exactly reverse order.

**Predicted output:**

```
enter countdown(3)
enter countdown(2)
enter countdown(1)
Blast off!
leave countdown(1)
leave countdown(2)
leave countdown(3)
```

In [ ]:
def countdown(n):
    if n == 0:                      # base case: the stopping condition
        print("Blast off!")
        return
    print(f"enter countdown({n})")  # plate goes ON the stack
    countdown(n - 1)                # this call waits here...
    print(f"leave countdown({n})")  # ...and resumes only now (plate comes OFF)

countdown(3)

### The stack, moment by moment

```
call countdown(3):   [ countdown(3) ]
call countdown(2):   [ countdown(3), countdown(2) ]
call countdown(1):   [ countdown(3), countdown(2), countdown(1) ]
call countdown(0):   [ countdown(3), countdown(2), countdown(1), countdown(0) ]   <- peak: 4 plates
return from (0):     [ countdown(3), countdown(2), countdown(1) ]
return from (1):     [ countdown(3), countdown(2) ]
return from (2):     [ countdown(3) ]
return from (3):     [ ]
```

**Recursion space = O(maximum stack depth).** Here that is O(n), even though the code creates no list at all. This is the classic hidden space cost interviewers probe.

## Demo 2 — Python's recursion limit

Python refuses to stack plates forever. The default ceiling is around **1000** frames. `countdown(5000)` would crash with `RecursionError: maximum recursion depth exceeded` — the logic is fine, the plate stack just hits the roof.

**What to observe:** the number printed below. **Predicted output:** `1000` (or a nearby value like 3000, depending on the environment).

Lesson: for very deep work, prefer a loop — a loop reuses one frame, O(1) auxiliary space.

In [ ]:
import sys
print(sys.getrecursionlimit())

## Demo 3 — `insert(0, x)` vs `append(x)`

A Python list keeps items in one continuous block, in order. Inserting at index 0 is like squeezing a passenger into **seat 1 of a full train-berth row**: every seated passenger must shift one seat right → **O(n)** per insert. `append` takes the free last seat → **O(1)**.

Doing `insert(0, ...)` inside a loop over n items is therefore an accidental **O(n²)**. The fix: `append` everything (O(1) each) and reverse once at the end, or use `collections.deque`, a double-ended queue built to accept items at both ends in O(1).

**What to observe:** all three approaches below build the SAME reversed list `[4, 3, 2, 1, 0]` — but only the first one pays the shifting cost at every step. On tiny n you will not feel it; at n = 100,000 the first way does ~10 billion shift operations while the other two stay linear.

**Predicted output:**

```
insert(0) way : [4, 3, 2, 1, 0]
append+reverse: [4, 3, 2, 1, 0]
deque way     : [4, 3, 2, 1, 0]
```

In [ ]:
from collections import deque

data = range(5)

slow = []                       # O(n^2): each insert shifts everything right
for x in data:
    slow.insert(0, x)

fast = []                       # O(n): appends are O(1), one reverse at the end
for x in data:
    fast.append(x)
fast.reverse()

d = deque()                     # O(n): appendleft is O(1), no shifting
for x in data:
    d.appendleft(x)

print("insert(0) way :", slow)
print("append+reverse:", fast)
print("deque way     :", list(d))

## Demo 4 — `in` on a list vs `in` on a set

`x in some_list` checks item by item — like finding a friend in a hotel by **knocking on every room door**: worst case n knocks, **O(n)**.

A **set** uses **hashing**: a math trick that turns the value into a number telling exactly which shelf it sits on. `x in some_set` is like asking the **reception register** — one lookup, **~O(1)** on average.

Trade-off: the set is a second copy of the data → **O(n) extra space**. Spending memory to buy speed — one of the most common moves in DSA.

**What to observe:** both answers are identical; only the *work behind the scenes* differs. The target is placed at the very END of the list — worst case for door-knocking. The set does not care how big n is.

**Predicted output:**

```
in list: True
in set : True
```

In [ ]:
n = 1_000_000
big_list = list(range(n))
big_set = set(big_list)          # one-time O(n) build, O(n) extra space

target = n - 1                   # worst case for the list: it is at the very end
print("in list:", target in big_list)   # walks up to n items -> O(n)
print("in set :", target in big_set)    # hash lookup -> ~O(1)

## Demo 5 — `chr()` and `ord()`: the tool for letter patterns

Every character has a standard code number (ASCII). `chr(code)` gives the character; `ord(char)` gives the code back. Capital letters sit at **65 = 'A'** through 90 = 'Z'. So "the i-th capital letter" is simply `chr(65 + i)`.

**What to observe:** codes 65–69 map to A–E, and `ord` undoes `chr`.

**Predicted output:**

```
i=0  code=65  chr=A  ord back=65
i=1  code=66  chr=B  ord back=66
i=2  code=67  chr=C  ord back=67
i=3  code=68  chr=D  ord back=68
i=4  code=69  chr=E  ord back=69
```

In [ ]:
for i in range(5):
    code = 65 + i
    ch = chr(code)
    print(f"i={i}  code={code}  chr={ch}  ord back={ord(ch)}")

## Practice — Patterns 16 to 22 (shapes + hints only, code is my job)

### Pattern 16 — Alphabet-repeat triangle (n = 5)

```
A
BB
CCC
DDDD
EEEEE
```

Hint: row `i` fixes ONE letter, `chr(65 + i)`, and repeats it `i + 1` times.

### Pattern 17 — Alphabet hill / palindrome pyramid (n = 5)

```
    A
   ABA
  ABCBA
 ABCDCBA
ABCDEDCBA
```

Hint: row `i` = `n - i - 1` spaces, then `2i + 1` letters — climb up while the column is at or before the middle, descend after it; the peak letter prints only once.

### Pattern 18 — Reverse-alphabet triangle (n = 5)

```
E
D E
C D E
B C D E
A B C D E
```

Hint: every row ENDS at `chr(64 + n)`; row `i` starts `i` letters earlier and counts upward.

### Pattern 19 — Hourglass of stars (n = 5)

```
**********
****  ****
***    ***
**      **
*        *
*        *
**      **
***    ***
****  ****
**********
```

Hint: every row is stars + middle spaces + stars; top half row `i` has `n - i` stars per side and `2i` spaces; the bottom half is the top half replayed in reverse.

### Pattern 20 — Butterfly (n = 5)

```
*        *
**      **
***    ***
****  ****
**********
****  ****
***    ***
**      **
*        *
```

Hint: mirror of 19 — wings grow to the full middle row, then shrink; with `i` stars per side the gap is `2 * (n - i)` spaces; `2n - 1` rows total.

### Pattern 21 — Hollow rectangle (n = 5)

```
*****
*   *
*   *
*   *
*****
```

Hint: `*` only when the cell is on a border — first/last row OR first/last column (one `if` with `or`s); space otherwise.

### Pattern 22 — Number rings (n = 5)

```
5 5 5 5 5 5 5 5 5
5 4 4 4 4 4 4 4 5
5 4 3 3 3 3 3 4 5
5 4 3 2 2 2 3 4 5
5 4 3 2 1 2 3 4 5
5 4 3 2 2 2 3 4 5
5 4 3 3 3 3 3 4 5
5 4 4 4 4 4 4 4 5
5 5 5 5 5 5 5 5 5
```

Hint: a `(2n-1) x (2n-1)` grid; each cell = `n` minus its distance from the NEAREST edge, i.e. `min(i, j, size-1-i, size-1-j)`.

## Practice — Three maths problems (approach + hints only)

### 1. Trailing zeroes in n!

How many zeroes at the end of `n! = 1 x 2 x ... x n`? Do **NOT** compute n! — it explodes in size.

- Each trailing zero is one factor of 10, and 10 = **2 x 5**. Count the (2, 5) pairs.
- 2s are everywhere; **5s are the rare ones** — so just count factors of 5 in 1..n.
- Formula idea: `n//5 + n//25 + n//125 + ...` until the term is 0.
- Why `//25` too? **25 = 5 x 5 carries TWO fives**; `n//5` counted it once, `n//25` adds its second five. 125 adds a third via `n//125`.
- Check yourself: n = 25 → `25//5 + 25//25 = 5 + 1 = 6` zeroes.

### 2. Digit sum

Sum the digits of a number (5341 → 13).

- `n % 10` peels the LAST digit; `n // 10` throws it away.
- Loop while `n > 0`: add the peel, then shrink. 5341 → 534 → 53 → 5 → 0.
- O(number of digits) time, O(1) auxiliary space.

### 3. Count of digits WITHOUT a loop

- String way: `len(str(n))` — mind the minus sign for negatives.
- Maths way: `floor(log10(n)) + 1`. Gentle idea: digit count jumps exactly at **powers of 10** — 1..9 have 1 digit, 10..99 have 2, 100..999 have 3. `log10` tells which power-of-10 band n falls in (log10 of 100..999 is 2.something → floor 2 → +1 = 3 digits).
- Edge cases for the log way: n = 0 (log undefined, answer is 1) and negatives (use `abs(n)` first).